# 06 Total-power term: pass and fail cases

This notebook isolates `PhysicsComponents.total_power`, the relative integrated-power mismatch between a predicted and a target profile, masked by a power floor. It confirms the term rewards mass agreement and is insensitive to where that mass sits in elevation.

Modules exercised: `PhysicsComponents.total_power`.

We engineer cases designed to pass (equal mass, possibly different shape) and to fail (scaled mass), and plot the term against a controlled amplitude ratio.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_default_dtype(torch.float32)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.titlesize" : 12,
    "axes.labelsize" : 11,
    "legend.fontsize": 9,
})

DEVICE = torch.device("cpu")


In [ ]:
from tools.gaussians                import GaussianMixture
from pipelines.training_pipeline.loss import PhysicsComponents

def to_check_tensor(profiles):
    t = torch.tensor(np.asarray(profiles, dtype=np.float32), device=DEVICE)
    return t.T.reshape(1, t.shape[1], 1, t.shape[0])

def gaussian_profile(x_axis, amp, mu, sigma):
    amps = np.asarray(amp,   dtype=np.float32).reshape(1, -1)
    mus  = np.asarray(mu,    dtype=np.float32).reshape(1, -1)
    sigs = np.asarray(sigma, dtype=np.float32).reshape(1, -1)
    return GaussianMixture.evaluate_batch(x_axis, amps, mus, sigs)[0]


In [ ]:
x_axis = np.linspace(-20.0, 40.0, 160).astype(np.float32)
dx     = float(x_axis[1] - x_axis[0])
floor  = 1e-3

def total_power(pred, target):
    return float(PhysicsComponents.total_power(to_check_tensor(pred[None, :]), to_check_tensor(target[None, :]), dx, floor))


## Pass case: equal mass, different shape

A narrow tall profile and a broad short profile can carry identical integrated power. The total-power term should be near zero because it only sees mass, not shape.

In [ ]:
target = gaussian_profile(x_axis, amp=1.0, mu=10.0, sigma=3.0)
mass_t = float(target.sum() * dx)

wide_sigma = 7.0
wide_amp   = mass_t / (wide_sigma * np.sqrt(2.0 * np.pi))
equal_mass = gaussian_profile(x_axis, amp=float(wide_amp), mu=22.0, sigma=wide_sigma)

fig, ax = plt.subplots(figsize=(6.6, 3.3))
ax.plot(x_axis, target,     color="k",  label=f"target, mass={mass_t:.2f}")
ax.plot(x_axis, equal_mass, color="C2", label=f"equal-mass, mass={float(equal_mass.sum()*dx):.2f}")
ax.set_xlabel("elevation [m]")
ax.set_ylabel("reflectivity")
ax.set_title(f"Equal-mass pass case, total_power = {total_power(equal_mass, target):.4f}")
ax.legend()
plt.show()

## Fail case: scaled mass

Scaling the amplitude scales the mass, which the term penalises linearly in the relative error.

In [ ]:
ratios = np.linspace(0.2, 2.5, 25)
vals   = [total_power(target * r, target) for r in ratios]

fig, ax = plt.subplots(figsize=(6.6, 3.4))
ax.plot(ratios, vals, "o-", color="C3")
ax.axvline(1.0, color="0.4", ls="--", lw=0.9, label="matched mass")
ax.set_xlabel("predicted / target amplitude")
ax.set_ylabel("total_power term")
ax.set_title("Total-power penalty vs mass ratio")
ax.legend()
plt.show()

print(f"total_power at ratio 1.0 : {total_power(target, target):.6f}")
print(f"total_power at ratio 2.0 : {total_power(target * 2.0, target):.6f}")

## Expected outcome

The equal-mass pair gives a total-power term at the floor despite very different shapes, confirming shape-invariance. The mass-ratio sweep is a clean V centred on ratio one, with the term equal to `|ratio - 1|`, confirming the term is a pure relative power penalty.